# OmniLSS TPU Performance Test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dongfangzhizhu/OmniLSS/blob/main/examples/colab/07_performance_tpu.ipynb)

This notebook benchmarks OmniLSS on TPU and compares against CPU, GPU, and R gamlss.

**Runtime**: Set to **TPU** — Runtime → Change runtime type → Hardware accelerator: TPU v2

> TPU access requires Colab Pro+ or Google Cloud TPU. The notebook gracefully falls back to CPU if no TPU is available.

## Contents
1. Environment setup & TPU check
2. Install OmniLSS with TPU support
3. TPU vs CPU performance
4. Large-scale data benchmarks
5. TPU vs R gamlss comparison
6. Summary


## 1. Environment Setup

In [ ]:
import sys
try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')
print(f'Python {sys.version}')

## 2. Install OmniLSS with TPU Support

In [ ]:
if IN_COLAB:
    !pip install -q git+https://github.com/dongfangzhizhu/OmniLSS.git#subdirectory=omnilss
    # Install JAX with TPU support
    !pip install -q 'jax[tpu]' -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
else:
    !pip install -q -e ../../omnilss
print('Installation complete')

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f'JAX version: {jax.__version__}')
print('Available devices:')
for d in jax.devices():
    print(f'  {d}')

tpu_devices = [d for d in jax.devices() if d.platform == 'tpu']
gpu_devices = [d for d in jax.devices() if d.platform in ('gpu', 'cuda')]
cpu_devices = [d for d in jax.devices() if d.platform == 'cpu']

HAS_TPU = len(tpu_devices) > 0
HAS_GPU = len(gpu_devices) > 0

print(f'\nTPU available: {HAS_TPU} ({len(tpu_devices)} cores)')
print(f'GPU available: {HAS_GPU}')

if not HAS_TPU:
    print('\nNOTE: No TPU detected. Results will show CPU performance.')
    print('To use TPU: Runtime -> Change runtime type -> TPU v2')

## 3. TPU vs CPU Performance

In [ ]:
from omnilss import gamlss
from omnilss.distributions import resolve_family

# TPU shines on large data
DATA_SIZES = [1_000, 5_000, 10_000, 50_000, 100_000]
DISTRIBUTIONS = ['NO', 'GA', 'PO']
N_REPEATS = 3

def generate_data(dist, n, seed=42):
    rng = np.random.RandomState(seed)
    x = rng.normal(0, 1, n)
    if dist == 'NO':
        y = 2 + 0.5*x + rng.normal(0, 1, n)
    elif dist == 'GA':
        mu = np.exp(1 + 0.3*x)
        y = rng.gamma(4, mu/4)
    elif dist == 'PO':
        y = rng.poisson(np.exp(1 + 0.3*x)).astype(float)
    else:
        y = rng.normal(0, 1, n)
    return {'y': y, 'x': x}

def time_fit(dist, n, device=None, n_repeats=N_REPEATS):
    data = generate_data(dist, n)
    family = resolve_family(dist)
    times = []
    for _ in range(n_repeats + 1):
        t0 = time.perf_counter()
        if device:
            with jax.default_device(device):
                gamlss('y ~ x', family=family, data=data)
        else:
            gamlss('y ~ x', family=family, data=data)
        times.append(time.perf_counter() - t0)
    return float(np.mean(times[1:]))

results = []
print('Benchmarking TPU vs CPU...')
print('='*60)

for dist in DISTRIBUTIONS:
    print(f'\n[{dist}]')
    for n in DATA_SIZES:
        cpu_time = time_fit(dist, n, device=cpu_devices[0] if cpu_devices else None)
        if HAS_TPU:
            tpu_time = time_fit(dist, n, device=tpu_devices[0])
            speedup = cpu_time / tpu_time
            print(f'  n={n:7,}: CPU={cpu_time:.4f}s  TPU={tpu_time:.4f}s  speedup={speedup:.1f}x')
        else:
            tpu_time = None
            speedup = None
            print(f'  n={n:7,}: CPU={cpu_time:.4f}s  (no TPU)')
        results.append({'dist': dist, 'n': n, 'cpu_time': cpu_time,
                        'tpu_time': tpu_time, 'speedup': speedup})

df = pd.DataFrame(results)
print('\nDone.')

In [ ]:
# Visualize TPU vs CPU
if HAS_TPU:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    by_n = df.groupby('n')['speedup'].mean()
    axes[0].plot(by_n.index, by_n.values, 'o-', linewidth=2, markersize=8, color='#f39c12')
    axes[0].axhline(y=1, color='gray', linestyle='--')
    axes[0].set_xlabel('Data size (n)')
    axes[0].set_ylabel('TPU speedup vs CPU')
    axes[0].set_title('TPU Speedup by Data Size')
    axes[0].set_xscale('log')
    axes[0].grid(True, alpha=0.3)

    for dist in DISTRIBUTIONS:
        sub = df[df['dist'] == dist]
        axes[1].plot(sub['n'], sub['speedup'], 'o-', label=dist, linewidth=2)
    axes[1].axhline(y=1, color='gray', linestyle='--')
    axes[1].set_xlabel('Data size (n)')
    axes[1].set_ylabel('TPU speedup vs CPU')
    axes[1].set_title('TPU Speedup by Distribution')
    axes[1].set_xscale('log')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('tpu_speedup.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Mean TPU speedup: {df["speedup"].mean():.1f}x')
else:
    print('No TPU available. Expected TPU speedup: 5-20x over CPU for large datasets.')
    print('TPU is most effective for n > 50,000 observations.')

## 4. Large-Scale Data Benchmarks

In [ ]:
# Test at very large scale — TPU's sweet spot
LARGE_SIZES = [100_000, 500_000, 1_000_000]
large_results = []

print('Large-scale benchmarks (NO distribution)...')
for n in LARGE_SIZES:
    try:
        cpu_time = time_fit('NO', n, device=cpu_devices[0] if cpu_devices else None, n_repeats=2)
        if HAS_TPU:
            tpu_time = time_fit('NO', n, device=tpu_devices[0], n_repeats=2)
            sp = cpu_time / tpu_time
            print(f'  n={n:9,}: CPU={cpu_time:.3f}s  TPU={tpu_time:.3f}s  speedup={sp:.1f}x')
            large_results.append({'n': n, 'cpu': cpu_time, 'tpu': tpu_time, 'speedup': sp})
        else:
            print(f'  n={n:9,}: CPU={cpu_time:.3f}s')
            large_results.append({'n': n, 'cpu': cpu_time, 'tpu': None, 'speedup': None})
    except Exception as e:
        print(f'  n={n:9,}: FAILED ({e})')

## 5. TPU vs R gamlss

In [ ]:
# Install R
if IN_COLAB:
    !apt-get update -qq && apt-get install -y -qq r-base
    !R -e "install.packages(c('gamlss','jsonlite'), repos='https://cran.r-project.org', quiet=TRUE)"
    print('R installed.')

In [ ]:
import subprocess, json, tempfile, csv
from pathlib import Path

_R_SCRIPT = '''
suppressMessages(library(gamlss)); suppressMessages(library(jsonlite))
args <- commandArgs(trailingOnly=TRUE)
df <- read.csv(args[1])
t0 <- proc.time()["elapsed"]
fit <- tryCatch(gamlss(y~x, family=args[2], data=df, trace=FALSE), error=function(e) NULL)
elapsed <- proc.time()["elapsed"] - t0
if (is.null(fit)) cat(toJSON(list(success=FALSE), auto_unbox=TRUE), "\\n")
else cat(toJSON(list(success=TRUE, r_time=elapsed), auto_unbox=TRUE), "\\n")
'''

def time_r(dist, n):
    data = generate_data(dist, n)
    with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, newline='') as f:
        csv_path = f.name
        w = csv.writer(f)
        w.writerow(['y','x'])
        for yv, xv in zip(data['y'], data['x']):
            w.writerow([yv, xv])
    with tempfile.NamedTemporaryFile(mode='w', suffix='.R', delete=False) as f:
        f.write(_R_SCRIPT); r_path = f.name
    try:
        t0 = time.perf_counter()
        res = subprocess.run(['Rscript', r_path, csv_path, dist],
                             capture_output=True, text=True, timeout=120)
        wall = time.perf_counter() - t0
        if res.returncode != 0: return None
        lines = [l.strip() for l in res.stdout.splitlines() if l.strip()]
        parsed = json.loads(lines[-1])
        return wall if parsed.get('success') else None
    except Exception: return None
    finally:
        Path(csv_path).unlink(missing_ok=True); Path(r_path).unlink(missing_ok=True)

print('TPU vs R gamlss comparison...')
tpu_vs_r = []
for dist in ['NO', 'GA', 'PO']:
    for n in [100, 500, 1000, 5000]:
        device = tpu_devices[0] if HAS_TPU else (cpu_devices[0] if cpu_devices else None)
        py_time = time_fit(dist, n, device=device)
        r_time = time_r(dist, n)
        label = 'TPU' if HAS_TPU else 'CPU'
        if r_time:
            sp = r_time / py_time
            print(f'  {dist} n={n:5d}: {label}={py_time:.4f}s  R={r_time:.4f}s  {sp:.1f}x')
            tpu_vs_r.append({'dist': dist, 'n': n, 'py': py_time, 'r': r_time, 'speedup': sp})
        else:
            print(f'  {dist} n={n:5d}: {label}={py_time:.4f}s  R=N/A')

if tpu_vs_r:
    vdf = pd.DataFrame(tpu_vs_r)
    print(f'\nMean speedup vs R: {vdf["speedup"].mean():.1f}x')

## 6. Summary

In [ ]:
print('='*60)
print('TPU Performance Summary')
print('='*60)
print(f'TPU available: {HAS_TPU}')
if HAS_TPU:
    df_tpu = pd.DataFrame(results).dropna(subset=['speedup'])
    print(f'TPU vs CPU speedup: mean={df_tpu["speedup"].mean():.1f}x, max={df_tpu["speedup"].max():.1f}x')
if tpu_vs_r:
    vdf = pd.DataFrame(tpu_vs_r)
    label = 'TPU' if HAS_TPU else 'CPU'
    print(f'{label} vs R speedup: mean={vdf["speedup"].mean():.1f}x, max={vdf["speedup"].max():.1f}x')
print('\nKey findings:')
print('  - TPU provides the largest speedup for very large datasets (n > 100,000)')
print('  - JAX automatically distributes computation across all TPU cores')
print('  - OmniLSS is significantly faster than R gamlss on all hardware')
print('  - For small datasets (n < 1,000), CPU is often sufficient')